# Giai đoạn 4: Giải PDE Heston bằng PINN (Physics-Informed Neural Network)

Notebook này huấn luyện một PINN 2 pha (Adam + L-BFGS) để xấp xỉ nghiệm của **phương trình vi phân riêng phần Heston (Heston PDE)**:

$$\frac{\partial V}{\partial \tau} - \frac{1}{2}s^2 v \frac{\partial^2 V}{\partial s^2} - \rho \xi s v \frac{\partial^2 V}{\partial s \partial v} - \frac{1}{2}\xi^2 v \frac{\partial^2 V}{\partial v^2} - rs\frac{\partial V}{\partial s} - \kappa(\theta - v)\frac{\partial V}{\partial v} + rV = 0$$

Trong đó $\tau = T - t$ là thời gian ngược (forward time).

---
### Kiến trúc Mạng
- **Input**: $(s, v, \tau)$ — giá tài sản chuẩn hoá, phương sai tức thời, thời gian đến đáo hạn
- **Output**: $V(s, v, \tau)$ — giá quyền chọn (chuẩn hoá)
- **Cấu trúc**: 6 lớp MLP × 128 neurons, hàm kích hoạt Tanh, khởi tạo Xavier

In [ ]:
import os, sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.optim as optim

sys.path.append(os.path.abspath('..'))
from src.models import HestonPINN, pde_residual, boundary_loss

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
np.random.seed(42)
print(f'Thiết bị: {DEVICE.upper()}')

In [ ]:
# Nạp tham số Heston + dữ liệu
with open('../data/processed/heston_parameters.json') as f:
    params = json.load(f)

kappa, theta, xi, rho, v0 = params['kappa'], params['theta'], params['xi'], params['rho'], params['v0']
r = 0.045

df = pd.read_csv('../data/processed/03_heston_monte_carlo_prices.csv')
df = df[df['price_heston_analytical'] > 5.0].copy().reset_index(drop=True)

S_ref = float(df['underlying_price_S'].mean())
K_ref = float(df['strike_K'].mean())
V_ref = float(df['price_heston_analytical'].max())

s_norm = (df['underlying_price_S'].values / S_ref).astype(np.float32)
v_var  = np.full(len(df), v0, dtype=np.float32)
v_data = df['time_to_maturity_T'].values.astype(np.float32)
V_data = (df['price_heston_analytical'].values / V_ref).astype(np.float32)

print(f'Dữ liệu: {len(df)} hợp đồng | S_ref={S_ref:.0f} USD | V_ref={V_ref:.0f} USD')

In [ ]:
# Khởi tạo model PINN
model = HestonPINN(hidden_layers=6, hidden_units=128).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Tham số: {total_params:,}')

N_PDE = 4096; N_BC = 2048
LAMBDA_BC = 10.0; LAMBDA_DATA = 5.0
loss_history = []

def sample_col():
    s = torch.rand(N_PDE, 1, device=DEVICE) * 2.0
    v = torch.rand(N_PDE, 1, device=DEVICE) * 2.0 * v0 + 1e-5
    t = torch.rand(N_PDE, 1, device=DEVICE)
    return s.requires_grad_(True), v.requires_grad_(True), t.requires_grad_(True)

def compute_loss():
    s_c, v_c, t_c = sample_col()
    l_pde = torch.mean(pde_residual(model, s_c, v_c, t_c, r, kappa, theta, xi, rho)**2)
    l_bc  = boundary_loss(model, 1.0, K_ref/S_ref, r, N_BC, DEVICE)
    s_t  = torch.tensor(s_norm, device=DEVICE).unsqueeze(1)
    v_t  = torch.tensor(v_var,  device=DEVICE).unsqueeze(1)
    tau_t= torch.tensor(v_data, device=DEVICE).unsqueeze(1)
    V_t  = torch.tensor(V_data, device=DEVICE).unsqueeze(1)
    l_data = torch.mean((model(s_t, v_t, tau_t) - V_t)**2)
    total = l_pde + LAMBDA_BC*l_bc + LAMBDA_DATA*l_data
    return total, l_pde, l_bc, l_data

In [ ]:
# PHA 1: Adam
ADAM_EPOCHS = 5000
opt = optim.Adam(model.parameters(), lr=1e-3)
sch = optim.lr_scheduler.StepLR(opt, step_size=1000, gamma=0.5)

t0 = time.time()
for ep in range(1, ADAM_EPOCHS+1):
    opt.zero_grad()
    loss, lp, lb, ld = compute_loss()
    loss.backward()
    opt.step(); sch.step()
    loss_history.append(loss.item())
    if ep % 500 == 0:
        print(f'[Adam] ep={ep:5d} | Total={loss.item():.5f} | PDE={lp.item():.5f} | BC={lb.item():.5f} | Data={ld.item():.5f}')

print(f'\n✅ Adam xong ({time.time()-t0:.1f}s) | Loss={loss_history[-1]:.6f}')

In [ ]:
# PHA 2: L-BFGS
LBFGS_ITERS = 500
opt_lb = optim.LBFGS(model.parameters(), lr=0.1, max_iter=20,
                     history_size=50, line_search_fn='strong_wolfe')
it = [0]

def closure():
    opt_lb.zero_grad()
    loss, _, _, _ = compute_loss()
    loss.backward()
    it[0] += 1
    loss_history.append(loss.item())
    if it[0] % 50 == 0:
        print(f'[L-BFGS] iter={it[0]:4d}/{LBFGS_ITERS} | Loss={loss.item():.6f}')
    return loss

for _ in range(LBFGS_ITERS // 20):
    opt_lb.step(closure)

print(f'\n✅ L-BFGS xong | Final Loss = {loss_history[-1]:.6f}')

In [ ]:
# Đường cong Loss
plt.figure(figsize=(10,5))
plt.semilogy(loss_history, color='royalblue', lw=1.2)
plt.axvline(x=ADAM_EPOCHS, color='red', ls='--', alpha=0.7, label='Adam → L-BFGS')
plt.xlabel('Iteration'); plt.ylabel('Loss (log scale)')
plt.title('PINN Training Loss — Heston PDE', fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3)
os.makedirs('../outputs/plots', exist_ok=True)
plt.savefig('../outputs/plots/pinn_loss_curve.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Đánh giá: PINN vs Giải tích
model.eval()
with torch.no_grad():
    s_t = torch.tensor(s_norm, device=DEVICE).unsqueeze(1)
    v_t = torch.tensor(v_var,  device=DEVICE).unsqueeze(1)
    tau_t= torch.tensor(v_data, device=DEVICE).unsqueeze(1)
    V_pinn = model(s_t, v_t, tau_t).cpu().numpy().flatten() * V_ref

V_anal = df['price_heston_analytical'].values
mae = np.mean(np.abs(V_pinn - V_anal))
mre = np.mean(np.abs(V_pinn - V_anal) / np.maximum(V_anal, 1.0)) * 100
r2  = 1 - np.sum((V_pinn - V_anal)**2) / np.sum((V_anal - V_anal.mean())**2)

print(f'MAE  = {mae:.4f} USD')
print(f'MRE  = {mre:.2f}%')
print(f'R²   = {r2:.6f}')

# Lưu kết quả
df['price_pinn'] = V_pinn
df['pinn_error_abs'] = np.abs(V_pinn - V_anal)
df['pinn_error_rel'] = df['pinn_error_abs'] / np.maximum(V_anal, 1.0)
df.to_csv('../data/processed/04_pinn_predictions.csv', index=False)

# Lưu checkpoint model
os.makedirs('../outputs/models', exist_ok=True)
torch.save({'model_state_dict': model.state_dict(),
            'hidden_layers': 6, 'hidden_units': 128,
            'S_ref': S_ref, 'V_ref': V_ref,
            'heston_params': params},
           '../outputs/models/heston_pinn.pt')
print('💾 Model saved to outputs/models/heston_pinn.pt')

In [ ]:
# Đồ thị so sánh
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(V_anal, V_pinn, alpha=0.4, s=12, c='steelblue')
lm = max(V_anal.max(), V_pinn.max())
axes[0].plot([0, lm], [0, lm], 'r--', lw=1.5, label='Perfect fit')
axes[0].set_xlabel('Analytical Price (USD)'); axes[0].set_ylabel('PINN Price (USD)')
axes[0].set_title('PINN vs. Analytical (Scatter)', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

rel_err = np.abs(V_pinn - V_anal) / np.maximum(V_anal, 1.0) * 100
axes[1].scatter(df['time_to_maturity_T']*365, rel_err, alpha=0.4, s=12, c='tomato')
axes[1].axhline(mre, color='orange', ls=':', lw=1.5, label=f'MRE avg ({mre:.2f}%)')
axes[1].set_xlabel('Days to Maturity'); axes[1].set_ylabel('Relative Error (%)')
axes[1].set_title('PINN Error by Maturity', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/plots/pinn_vs_analytical.png', dpi=300, bbox_inches='tight')
plt.show()